# SecAlign Defense: Paper-Exact Dataset Generation, Preference Optimization (DPO/ORPO) & Benchmark Evaluation

This notebook replicates the **exact experimental workflow** from the official paper repository:
**"SecAlign: Defending Against Prompt Injection with Preference Optimization"** (Sizhe Chen, Arman Zharmagambetov, Saeed Mahloujifar, Kamalika Chaudhuri, David Wagner, Chuan Guo — CCS 2025 / arXiv:2410.05451).

## 📌 Key Paper Specifications
1. **Datasets**: Downloads original `alpaca_data_cleaned.json` (51.7k samples) and `alpaca_data.json` for reference output completions.
2. **Delimiters & Randomization**: Implements `format_with_other_delimiters` to randomize delimiters during training (making SecAlign robust against unseen delimiter formats).
3. **Attack Generation**: Uses `NaiveCompletion` attack construction (combining 90% naive prompt injection with 10% adversarial delimiter completion overrides).
4. **Fine-Tuning**: Performs DPO / ORPO preference optimization with PEFT LoRA (`r=64`, `lora_alpha=8`, `lora_dropout=0.1`, `beta=0.1`, `epochs=3`).
5. **Benchmark Evaluation**: Evaluates the trained model via `SecAlignDefense` against standard prompt injection attacks in `ipi`.

In [ ]:
# Cell 1 — Installation & Environment Setup
# Install ipi benchmark package and fine-tuning dependencies
!pip install -q git+https://github.com/alirezaAalaie/IPI-Aaptive.git
!pip install -q trl peft transformers datasets bitsandbytes accelerate

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# Cell 2 — Download Original SecAlign Paper Datasets
import os, urllib.request

os.makedirs("./data", exist_ok=True)

# Exact URLs from SecAlign setup.py
DATA_URLS = {
    "alpaca_data_cleaned.json": "https://raw.githubusercontent.com/gururise/AlpacaDataCleaned/refs/heads/main/alpaca_data_cleaned.json",
    "alpaca_data.json": "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/refs/heads/main/alpaca_data.json"
}

for filename, url in DATA_URLS.items():
    dest_path = os.path.join("./data", filename)
    if not os.path.exists(dest_path):
        print(f"Downloading {filename} from {url}...")
        urllib.request.urlretrieve(url, dest_path)
        print(f"Saved {dest_path}")
    else:
        print(f"{dest_path} already exists.")


In [ ]:
# Cell 3 — Generate SecAlign Preference Data (Exact Paper Algorithm)
from ipi.defenses.secalign import load_paper_datasets, generate_secalign_preference_data

# Load paper datasets
clean_data, ref_inst_resp = load_paper_datasets(data_dir="./data")
print(f"Loaded {len(clean_data)} clean samples from alpaca_data_cleaned.json")
print(f"Loaded {len(ref_inst_resp)} reference responses from alpaca_data.json")

# Generate SecAlign preference pairs using paper's exact NaiveCompletion attack &
# format_with_other_delimiters randomization
pref_data = generate_secalign_preference_data(
    clean_data=clean_data,
    ref_inst_resp=ref_inst_resp,
    frontend_delimiters="TextTextText",   # Or 'Meta-Llama-3-8B-Instruct'
    attack="NaiveCompletion",            # Paper's default defensive alignment attack
    alignment="dpo",                     # DPO alignment
    seed=42,
    max_samples=1000                     # Adjust as needed for fast test or full 50k training
)

print(f"\n✅ Successfully generated {len(pref_data)} paper-exact preference pairs!\n")
print("--- SAMPLE PAPER PREFERENCE PAIR ---")
print("PROMPT:\n", pref_data[0]["prompt"])
print("CHOSEN (Clean Goal):\n", pref_data[0]["chosen"])
print("REJECTED (Injected Goal):\n", pref_data[0]["rejected"])


In [ ]:
# Cell 4 — Fine-Tune SecAlign via DPO/ORPO on Kaggle T4 GPU
from ipi.defenses.secalign import train_secalign

# Model parameters from SecAlign paper:
#  - Meta-Llama-3-8B-Instruct (dpo_lr = 1.6e-4)
#  - Mistral-7B-Instruct-v0.1 (dpo_lr = 1.4e-4)
#  - Qwen/Qwen2.5-1.5B-Instruct or Qwen2.5-3B-Instruct (dpo_lr = 2.0e-4)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./secalign_paper_model"

# Execute full paper fine-tuning (Uncomment below to run on GPU kernel)
"""
trainer = train_secalign(
    model_name_or_path=MODEL_ID,
    output_dir=OUTPUT_DIR,
    clean_data=clean_data,
    attack="NaiveCompletion",
    alignment="dpo",
    use_4bit=True,                   # Quantized NF4 QLoRA for Kaggle T4 VRAM efficiency
    lora_r=64,                       # Paper exact LoRA rank r=64
    lora_alpha=8,                    # Paper exact lora_alpha=8
    learning_rate=2.0e-4,            # Paper exact DPO learning rate
    num_train_epochs=3.0,            # Paper exact 3 epochs
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    max_samples=2000                 # Pass None for full 50k training
)
print(f"✅ Training complete! Model saved to {OUTPUT_DIR}")
"""


In [ ]:
# Cell 5 — Save & Publish Model Adapter to Hugging Face / Kaggle Datasets
"""
# Push model adapter to Hugging Face Hub:
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id="your-username/SecAlign-Qwen2.5-1.5B-DPO",
    repo_type="model"
)
"""
print("Model publishing pipeline ready.")


In [ ]:
# Cell 6 — Evaluate SecAlign Defense on ipi Attack Benchmark
from ipi.target import LocalLLM
from ipi.defenses.secalign import SecAlignDefense
from ipi.attacks import NaiveAttacker, IgnoreAttacker, FakeCompletionAttacker
from ipi.evaluator import BipiaSuccessEvaluator
from ipi.dataset import HijackDataset

"""
# 1. Initialize defended target
base_victim = LocalLLM(
    model_name_or_path=MODEL_ID,
    adapter_path=OUTPUT_DIR
)
secalign_victim = SecAlignDefense(target=base_victim, frontend_delimiters="TextTextText")

# 2. Initialize evaluation benchmark
dataset = HijackDataset(limit=10)
evaluator = BipiaSuccessEvaluator()
attackers = [NaiveAttacker(), IgnoreAttacker(), FakeCompletionAttacker()]

# 3. Run Benchmark Evaluation
for attacker in attackers:
    print(f"Evaluating attack: {attacker.__class__.__name__}...")
"""
print("SecAlign benchmark evaluation pipeline configured.")
